In [ ]:
# --- Environment cleanup (run this FIRST) -------------------------------------
# AutoMLConfig scans every installed package for its version. Interrupted
# `%pip install` commands can leave corrupted/partial package folders in
# site-packages (names starting with '~'), which have no 'Version' metadata and
# cause `KeyError: 'Version'` when building AutoMLConfig. This cell finds and
# removes them.
#
# IMPORTANT: after running this cell, RESTART THE KERNEL (Kernel > Restart),
# then run the notebook from the next cell onward.
import os, shutil, site, glob
import importlib.metadata as ilm

print("Scanning for broken distribution metadata...")
for dist in ilm.distributions():
    try:
        if dist.metadata["Version"] is None:
            print("  MISSING Version:", dist.metadata["Name"], "->", dist._path)
    except Exception as e:
        print("  UNREADABLE:", getattr(dist, "_path", "?"), "->", e)

paths = list(site.getsitepackages())
try:
    paths.append(site.getusersitepackages())
except Exception:
    pass

for sp in paths:
    if os.path.isdir(sp):
        for junk in glob.glob(os.path.join(sp, "~*")):
            print("Removing corrupted leftover:", junk)
            shutil.rmtree(junk, ignore_errors=True)

print("\nDone. Now RESTART THE KERNEL, then run from the next cell onward.")


In [ ]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

In [ ]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = "project-cluster"

# Create compute cluster (reuse it if it already exists).
# Use vm_size = "Standard_D2_V2" and max_nodes no greater than 4.
try:
    compute_target = ComputeTarget(workspace=ws, name=cluster_name)
    print("Found existing compute target.")
except ComputeTargetException:
    print("Creating a new compute target...")
    compute_config = AmlCompute.provisioning_configuration(
        vm_size="Standard_D2_V2",
        max_nodes=4,
    )
    compute_target = ComputeTarget.create(ws, cluster_name, compute_config)

# Wait until the cluster is ready to accept jobs.
compute_target.wait_for_completion(show_output=True)
print("Compute target ready:", compute_target.name)


In [ ]:
from azureml.widgets import RunDetails
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os
import shutil

# Specify parameter sampler
ps = RandomParameterSampling({
    "--C": uniform(0.08, 0.1),                 # Inverse regularization strength
    "--max_iter": choice(25, 50, 100, 200),    # Maximum iterations
})

# Specify a Policy
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1, delay_evaluation=5)

if "training" not in os.listdir():
    os.mkdir("./training")

# Make the training script available inside the source directory.
shutil.copy("train.py", "./training")

# Setup environment for your training run
sklearn_env = Environment.from_conda_specification(name='sklearn-env', file_path='conda_dependencies.yml')

# Create a ScriptRunConfig Object to specify the configuration details of your training job
src = ScriptRunConfig(
    source_directory="./training",
    script="train.py",
    compute_target=compute_target,
    environment=sklearn_env,
)

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
hyperdrive_config = HyperDriveConfig(
    run_config=src,
    hyperparameter_sampling=ps,
    policy=policy,
    primary_metric_name="Accuracy",
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs=20,
    max_concurrent_runs=4,
)


In [ ]:
# Submit your hyperdrive run to the experiment and show run details with the widget.
hyperdrive_run = exp.submit(hyperdrive_config)
RunDetails(hyperdrive_run).show()
hyperdrive_run.wait_for_completion(show_output=True)


In [ ]:
import joblib
# Get your best run and save the model from that run.
best_run = hyperdrive_run.get_best_run_by_primary_metric()

print("Best Run Id:", best_run.id)
print("Metrics:", best_run.get_metrics())
print("Files:", best_run.get_file_names())

# Register the model that train.py saved to outputs/hyperdrive_model.joblib
best_model = best_run.register_model(
    model_name="hyperdrive_model",
    model_path="outputs/hyperdrive_model.joblib",
)
print("Registered model:", best_model.name, "version", best_model.version)


In [ ]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
data_url = "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"
ds = TabularDatasetFactory.from_delimited_files(path=data_url)


In [ ]:
from train import clean_data
from sklearn.model_selection import train_test_split
from azureml.core import Dataset

# Use the clean_data function to clean your data.
x, y = clean_data(ds)

# AutoML expects a single labelled dataset. Build it, then register it as a
# TabularDataset so AutoML can run on the REMOTE compute cluster. Running AutoML
# remotely uses a managed environment with compatible package versions, which
# avoids the local Python 3.10 package-compatibility errors.
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
train_data = x_train.copy()
train_data["y"] = y_train

datastore = ws.get_default_datastore()
train_ds = Dataset.Tabular.register_pandas_dataframe(
    dataframe=train_data,
    target=(datastore, "bankmarketing_clean"),
    name="bankmarketing_clean",
    show_progress=True,
)


In [ ]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# Running AutoML on the remote compute cluster (compute_target) uses a managed
# environment with compatible package versions, avoiding the local Python 3.10
# package-compatibility errors.
automl_config = AutoMLConfig(
    compute_target=compute_target,
    experiment_timeout_minutes=30,
    task='classification',
    primary_metric='accuracy',
    training_data=train_ds,
    label_column_name='y',
    n_cross_validations=5,
)


In [ ]:
# Submit your automl run
automl_run = exp.submit(automl_config, show_output=True)


In [ ]:
# Retrieve and save your best automl model.
import joblib

best_automl_run, fitted_model = automl_run.get_output()

print("Best AutoML run:", best_automl_run.id)
print(fitted_model)

# Save the fitted model locally and register it in the workspace.
joblib.dump(fitted_model, "automl_best_model.joblib")
best_automl_run.register_model(
    model_name="automl_model",
    model_path="outputs/model.pkl",
)


In [ ]:
# Clean up the compute cluster to stop incurring charges.
compute_target.delete()
print("Compute cluster deletion requested.")
